In [5]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import pystac_client
import planetary_computer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

In [70]:
hh = pd.read_csv("../data/target_data_pipeline.csv")

# Create proper monthly datetime column
hh["month"] = pd.to_datetime(
    dict(year=hh["year"], month=hh["month"], day=1)
)

# Filter baseline period
hh = hh[
    (hh["month"] >= "2008-01-01") &
    (hh["month"] <= "2016-12-31")
]

print("Rows after filtering:", len(hh))
print("Unique households:", hh["hhid"].nunique())

Rows after filtering: 49796
Unique households: 1022


In [73]:
hh = pd.read_csv("../data/target_data_pipeline.csv")

hh["month"] = pd.to_datetime(
    dict(year=hh["year"], month=hh["month"], day=1)
)
print("Full HH range:",
      hh["month"].min(),
      hh["month"].max())

print("Rows per year:")
print(hh["month"].dt.year.value_counts().sort_index())

Full HH range: 2007-07-01 00:00:00 2020-12-01 00:00:00
Rows per year:
month
2007       2
2008     773
2009    7603
2010    9235
2011    8932
2012    7854
2013    6310
2014    5447
2015    3642
2019    1709
2020    2846
Name: count, dtype: int64


In [8]:
# Ensure coordinates numeric
hh["gps_longitude"] = pd.to_numeric(hh["gps_longitude"], errors="coerce")
hh["gps_latitude"] = pd.to_numeric(hh["gps_latitude"], errors="coerce")

hh = hh.dropna(subset=["gps_longitude", "gps_latitude"])

hh_unique = hh[['hhid','gps_longitude','gps_latitude']].drop_duplicates()

hh_gdf = gpd.GeoDataFrame(
    hh_unique,
    geometry=gpd.points_from_xy(
        hh_unique.gps_longitude,
        hh_unique.gps_latitude
    ),
    crs="EPSG:4326"
)

In [9]:
lon_vals = hh_gdf.geometry.x.values
lat_vals = hh_gdf.geometry.y.values

MY_BBOX = [
    float(lon_vals.min()),
    float(lat_vals.min()),
    float(lon_vals.max()),
    float(lat_vals.max()),
]

In [10]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

collection = catalog.get_collection("terraclimate")
asset = collection.assets["zarr-abfs"]
open_kwargs = asset.extra_fields["xarray:open_kwargs"]

ds_tc = xr.open_dataset(asset.href, **open_kwargs)

ds_tc = ds_tc.sel(
    lon=slice(MY_BBOX[0], MY_BBOX[2]),
    lat=slice(MY_BBOX[3], MY_BBOX[1]),
    time=slice("2008-01-01", "2016-12-31")
)

ds_tc = ds_tc[["soil", "ppt", "pdsi", "vpd"]].load()

In [11]:
tc_dfs = []

for i, (idx, row) in enumerate(hh_gdf.iterrows()):

    if i % 100 == 0:
        print(f"Processed {i} households")

    hhid = row['hhid']
    lon = row.geometry.x
    lat = row.geometry.y

    lon_idx = np.abs(ds_tc.lon.values - lon).argmin()
    lat_idx = np.abs(ds_tc.lat.values - lat).argmin()

    df = pd.DataFrame({
        "hhid": hhid,
        "month": pd.to_datetime(ds_tc.time.values),
        "soil": ds_tc.soil[:, lat_idx, lon_idx].values,
        "ppt": ds_tc.ppt[:, lat_idx, lon_idx].values,
        "pdsi": ds_tc.pdsi[:, lat_idx, lon_idx].values,
        "vpd": ds_tc.vpd[:, lat_idx, lon_idx].values
    })

    tc_dfs.append(df)

df_tc = pd.concat(tc_dfs, ignore_index=True)

Processed 0 households
Processed 100 households
Processed 200 households
Processed 300 households
Processed 400 households
Processed 500 households
Processed 600 households
Processed 700 households
Processed 800 households
Processed 900 households
Processed 1000 households


In [13]:
os.makedirs("../data/processed", exist_ok=True)

df_tc.to_parquet(
    "../data/processed/terraclimate_household_monthly_2008_2016.parquet",
    index=False
)